In [3]:
import torch


# Simple module for demonstration
class MyModule(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.param = torch.nn.Parameter(torch.rand(3, 4))
        self.linear = torch.nn.Linear(4, 5)

    def forward(self, x):
        return self.linear(x + self.param).clamp(min=0.0, max=1.0)


module = MyModule()

from torch.fx import symbolic_trace

# Symbolic tracing frontend - captures the semantics of the module
symbolic_traced: torch.fx.GraphModule = symbolic_trace(module)

# High-level intermediate representation (IR) - Graph representation
print(symbolic_traced.graph)
"""
graph():
    %x : [num_users=1] = placeholder[target=x]
    %param : [num_users=1] = get_attr[target=param]
    %add : [num_users=1] = call_function[target=operator.add](args = (%x, %param), kwargs = {})
    %linear : [num_users=1] = call_module[target=linear](args = (%add,), kwargs = {})
    %clamp : [num_users=1] = call_method[target=clamp](args = (%linear,), kwargs = {min: 0.0, max: 1.0})
    return clamp
"""

# Code generation - valid Python code
print(symbolic_traced.code)
"""
def forward(self, x):
    param = self.param
    add = x + param;  x = param = None
    linear = self.linear(add);  add = None
    clamp = linear.clamp(min = 0.0, max = 1.0);  linear = None
    return clamp
"""

graph():
    %x : [num_users=1] = placeholder[target=x]
    %param : [num_users=1] = get_attr[target=param]
    %add : [num_users=1] = call_function[target=operator.add](args = (%x, %param), kwargs = {})
    %linear : [num_users=1] = call_module[target=linear](args = (%add,), kwargs = {})
    %clamp : [num_users=1] = call_method[target=clamp](args = (%linear,), kwargs = {min: 0.0, max: 1.0})
    return clamp



def forward(self, x):
    param = self.param
    add = x + param;  x = param = None
    linear = self.linear(add);  add = None
    clamp = linear.clamp(min = 0.0, max = 1.0);  linear = None
    return clamp
    


'\ndef forward(self, x):\n    param = self.param\n    add = x + param;  x = param = None\n    linear = self.linear(add);  add = None\n    clamp = linear.clamp(min = 0.0, max = 1.0);  linear = None\n    return clamp\n'

In [5]:
from torch.fx.passes.shape_prop import ShapeProp

example_x = torch.rand(3, 4)

ShapeProp(symbolic_traced).propagate(example_x)

for node in symbolic_traced.graph.nodes:
    print(node.name, node.op, node.target, node.meta.get("tensor_meta"))
    print()

x placeholder x TensorMetadata(shape=torch.Size([3, 4]), dtype=torch.float32, requires_grad=False, stride=(4, 1), memory_format=torch.contiguous_format, is_quantized=False, qparams={})

param get_attr param TensorMetadata(shape=torch.Size([3, 4]), dtype=torch.float32, requires_grad=True, stride=(4, 1), memory_format=torch.contiguous_format, is_quantized=False, qparams={})

add call_function <built-in function add> TensorMetadata(shape=torch.Size([3, 4]), dtype=torch.float32, requires_grad=True, stride=(4, 1), memory_format=torch.contiguous_format, is_quantized=False, qparams={})

linear call_module linear TensorMetadata(shape=torch.Size([3, 5]), dtype=torch.float32, requires_grad=True, stride=(5, 1), memory_format=torch.contiguous_format, is_quantized=False, qparams={})

clamp call_method clamp TensorMetadata(shape=torch.Size([3, 5]), dtype=torch.float32, requires_grad=True, stride=(5, 1), memory_format=torch.contiguous_format, is_quantized=False, qparams={})

output output output Tens

In [6]:
import sys
sys.path.insert(0, ".")

import torch
import torch.nn as nn
from torch.fx import symbolic_trace
from torch.fx.passes.shape_prop import ShapeProp

from agent_shaper.transformer.model import GPT, GPTConfig, LayerNorm, CausalSelfAttention, MLP, Block

# Small config so the full GPT run is fast
cfg = GPTConfig(
    block_size=32,
    vocab_size=256,
    n_layer=2,
    n_head=2,
    n_embd=64,
    dropout=0.0,
    bias=True,
)

B, T, C = 2, 16, cfg.n_embd  # batch, seq_len, embed_dim

# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────

def _shapes_from_output(out):
    if isinstance(out, torch.Tensor):
        return (tuple(out.shape),)
    if isinstance(out, (tuple, list)):
        return tuple(tuple(t.shape) for t in out if isinstance(t, torch.Tensor))
    return ()

def _run_hooks(name, module, example_args):
    """Forward-hook based shape capture for modules that can't be symbolically traced."""
    records = []
    handles = []

    def make_hook(label):
        def hook(mod, inp, out):
            in_shapes = [tuple(t.shape) for t in inp if isinstance(t, torch.Tensor)]
            out_shapes = _shapes_from_output(out)
            records.append((label, type(mod).__name__, in_shapes, out_shapes))
        return hook

    # Root module hook
    handles.append(module.register_forward_hook(make_hook(f"[{name}]")))
    for subname, submod in module.named_modules():
        if subname:
            handles.append(submod.register_forward_hook(make_hook(subname)))

    with torch.no_grad():
        module(*example_args)

    for h in handles:
        h.remove()

    # Only print entries that actually fired (had a forward call)
    for label, cls, in_s, out_s in records:
        print(f"  {label:45s}  {cls:25s}  in={in_s}  ->  out={out_s}")


def run_shape_prop(name, module, example_args):
    SEP = "═" * 78
    print(f"\n{SEP}")
    print(f"  {name}")
    print(SEP)

    try:
        traced = symbolic_trace(module)
        ShapeProp(traced).propagate(*example_args)
        for node in traced.graph.nodes:
            meta = node.meta.get("tensor_meta")
            if meta is not None:
                print(
                    f"  {node.name:30s}  op={node.op:18s}"
                    f"  shape={tuple(meta.shape)}  dtype={meta.dtype}"
                )
            else:
                print(f"  {node.name:30s}  op={node.op:18s}  (no tensor meta)")
        print("  [method: symbolic_trace + ShapeProp]")

    except Exception as exc:
        print(f"  symbolic_trace failed → {type(exc).__name__}: {exc}")
        print("  falling back to forward hooks:\n")
        _run_hooks(name, module, example_args)

# ──────────────────────────────────────────────────────────────────────────────
# Run ShapeProp / hooks for every module in model.py + the top-level GPT
# ──────────────────────────────────────────────────────────────────────────────

modules = [
    # (display name,        module instance,           example inputs)
    ("LayerNorm",           LayerNorm(C, bias=True),        (torch.rand(B, T, C),)),
    ("MLP",                 MLP(cfg),                       (torch.rand(B, T, C),)),
    ("Block",               Block(cfg),                     (torch.rand(B, T, C),)),
    # CausalSelfAttention: B,T,C = x.size() unpacks a Proxy → TraceError
    ("CausalSelfAttention", CausalSelfAttention(cfg),       (torch.rand(B, T, C),)),
    # GPT: b,t = idx.size() + assert on Proxy → TraceError
    ("GPT (full model)",    GPT(cfg),                       (torch.randint(0, cfg.vocab_size, (B, T)),)),
]

for mod_name, mod, args in modules:
    mod.eval()
    run_shape_prop(mod_name, mod, args)


number of parameters: 0.12M

══════════════════════════════════════════════════════════════════════════════
  LayerNorm
══════════════════════════════════════════════════════════════════════════════
  input_1                         op=placeholder         shape=(2, 16, 64)  dtype=torch.float32
  weight                          op=get_attr            shape=(64,)  dtype=torch.float32
  bias                            op=get_attr            shape=(64,)  dtype=torch.float32
  getattr_1                       op=call_function       (no tensor meta)
  layer_norm                      op=call_function       shape=(2, 16, 64)  dtype=torch.float32
  output                          op=output              shape=(2, 16, 64)  dtype=torch.float32
  [method: symbolic_trace + ShapeProp]

══════════════════════════════════════════════════════════════════════════════
  MLP
══════════════════════════════════════════════════════════════════════════════
  x                               op=placeholder       